### Задание 1
Есть запрос query и несколько вариантов ответа для него `candidates`. Реализуйте функцию `rerank`, которая будет оценивать ответы на вопрос и ранжировать их от наиболее близкого к менее. 

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import torch


model_name = "cross-encoder/stsb-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name) #.to("cuda")


query = "How to choose a laptop for work?"
candidates = [
"2023 rating of best laptops for office work",
"Comparison of Intel Core i5 vs i7 processors",
"How to improve performance of an old laptop",
"Optimal laptop specifications for programmers",
"Difference between SSD and HDD drives",
"10 common mistakes when buying a laptop",
"How to connect a laptop to a TV",
"Best budget laptops under 50,000 rubles",
"What graphics card is needed for graphic design work",
"How to extend laptop battery life",
]


def rerank(query, candidates):
    results = []

    # передаём каждую пару совместно
    for candidate in candidates:
        inputs = tokenizer(query, candidate, return_tensors="pt", padding=True, truncation=True) #.to("cuda")
        with torch.no_grad():
            outputs = model(**inputs)

        # получаем вероятность и округляем
        score = torch.sigmoid(outputs.logits).item()
        results.append((candidate, round(score, 4)))
    return sorted(results, key=lambda x: x[1], reverse=True)


reranked = rerank(query, candidates)
reranked

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9808.21it/s]


[('How to connect a laptop to a TV', 0.3942),
 ('2023 rating of best laptops for office work', 0.3703),
 ('Optimal laptop specifications for programmers', 0.3702),
 ('How to improve performance of an old laptop', 0.3137),
 ('How to extend laptop battery life', 0.2821),
 ('10 common mistakes when buying a laptop', 0.2469),
 ('Best budget laptops under 50,000 rubles', 0.2292),
 ('What graphics card is needed for graphic design work', 0.1622),
 ('Comparison of Intel Core i5 vs i7 processors', 0.0722),
 ('Difference between SSD and HDD drives', 0.0295)]

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def format_instruction(instruction, query, doc):
    if instruction is None:
        instruction = 'Given a web search query, retrieve relevant passages that answer the query'
    output = "<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}".format(instruction=instruction,query=query, doc=doc)
    return output

def process_inputs(pairs):
    inputs = tokenizer(
        pairs, padding=False, truncation='longest_first',
        return_attention_mask=False, max_length=max_length - len(prefix_tokens) - len(suffix_tokens)
    )
    for i, ele in enumerate(inputs['input_ids']):
        inputs['input_ids'][i] = prefix_tokens + ele + suffix_tokens
    inputs = tokenizer.pad(inputs, padding=True, return_tensors="pt", max_length=max_length)

    # переносим тензоры на девайс модели
    for key in inputs:
        inputs[key] = inputs[key].to(model.device)
    return inputs

@torch.no_grad()
def compute_logits(inputs):
    batch_scores = model(**inputs).logits[:, -1, :]
    true_vector = batch_scores[:, token_true_id]
    false_vector = batch_scores[:, token_false_id]
    batch_scores = torch.stack([false_vector, true_vector], dim=1)
    batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)
    scores = batch_scores[:, 1].exp().tolist()
    return scores

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Reranker-0.6B", padding_side='left')
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-Reranker-0.6B").eval()

token_false_id = tokenizer.convert_tokens_to_ids("no")
token_true_id = tokenizer.convert_tokens_to_ids("yes")
max_length = 8192

prefix = "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)

task = 'Given a web search query, retrieve relevant passages that answer the query'

queries = ['Сколько лететь до марса',
    'растёт ли кукуруза в тени',
]

documents = [
   "Время полёта на Марс зависит от множества факторов, включая траекторию, скорость корабля и расположение планет. В среднем полёт может занять от 6 до 9 месяцев. Самый быстрый способ добраться до Марса, по расчётам, займёт около 70-80 суток, но потребует значительного количества топлива",
   "Кукуруза не растёт хорошо в тени. Она нуждается в достаточном количестве солнечного света для нормального развития и плодоношения",
   "Уход за кукурузой включает в себя полив, рыхление, прополку, подкормку и удаление пасынков. Важно обеспечить кукурузе достаточное количество влаги, особенно в период цветения и формирования початков, а также поддерживать почву рыхлой и свободной от сорняков."
]

pairs = []
for q in queries:
    for d in documents:
        pairs.append(format_instruction(task, q, d))

inputs = process_inputs(pairs)
scores = compute_logits(inputs)

print("scores: ", [round(x, 6) for x in scores])

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 7138.26it/s]
/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 7/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2356: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


scores:  [1.0, 9e-06, 3e-06, 4e-06, 1.0, 0.890625]
